In [2]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")          # try 'yolo11n.pt' if you need more speed
results = model("images/traffic1.jpg", save=True)  # writes runs/detect/predict/ with a drawn image

# Optional: inspect detections
r = results[0]
for c, conf, box in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy):
    print(int(c), float(conf), box.tolist())



image 1/1 s:\Dokumenty\Code\Heimdall\images\traffic1.jpg: 448x640 1 person, 23 cars, 4 trucks, 298.0ms
Speed: 14.1ms preprocess, 298.0ms inference, 19.5ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs\detect\predict
2 0.9104148745536804 [1094.4661865234375, 2789.722900390625, 1924.4051513671875, 3510.6259765625]
2 0.8977630138397217 [608.8037719726562, 2422.369140625, 1364.95068359375, 3008.8505859375]
2 0.8845630884170532 [2466.528076171875, 2896.7080078125, 3203.843994140625, 3581.337646484375]
2 0.8736421465873718 [1524.503173828125, 1948.0040283203125, 2086.6640625, 2373.971435546875]
2 0.8650481700897217 [2957.646240234375, 2128.431640625, 3564.791259765625, 2633.685791015625]
2 0.857619047164917 [1989.1611328125, 2040.1961669921875, 2636.79736328125, 2623.5947265625]
2 0.8542324304580688 [1983.0145263671875, 1641.0167236328125, 2489.497314453125, 2006.559326171875]
2 0.8484529256820679 [2509.210205078125, 1770.2713623046875, 3059.498779296875, 2238.674560

## Video analysis test

on a 20s grainy highway video

In [5]:
import os
import csv
import time
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO
import yaml

# --------- Config ---------
VIDEO_IN = "videos/british_highway_traffic.mp4"
MODEL_WEIGHTS = "yolo11s.pt"     # try yolo11m.pt if you have GPU headroom
TRACKER = "botsort.yaml"         # or "bytetrack.yaml"
CONF = 0.25
IOU = 0.5

# COCO vehicle classes: 2 car, 3 motorcycle, 5 bus, 7 truck
VEHICLE_CLASS_IDS = [2, 3, 5, 7]
CLASS_LABELS = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

# Virtual counting line (edit these two points for your scene)
# Format: (x, y) in pixel coords of the ORIGINAL video frames
LINE_P1 = (200, 300)
LINE_P2 = (900, 300)

OUT_DIR = "runs/traffic"
os.makedirs(OUT_DIR, exist_ok=True)
idx = len(os.listdir(OUT_DIR)) // 2
VIDEO_OUT = os.path.join(OUT_DIR, f"annotated_{idx}.mp4")
CSV_OUT = os.path.join(OUT_DIR, f"events_{idx}.csv")

# --------- Helpers ---------
def line_side(px, py, ax, ay, bx, by):
    # cross product sign: >0 one side, <0 other side, ==0 on line
    return (bx - ax) * (py - ay) - (by - ay) * (px - ax)

def draw_line_with_count(frame, p1, p2, count_a_to_b, count_b_to_a):
    cv2.line(frame, p1, p2, (0, 255, 255), 2)
    # label box
    label = f"A→B: {count_a_to_b} | B→A: {count_b_to_a}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (10, 10), (20 + tw, 20 + th), (0, 0, 0), -1)
    cv2.putText(frame, label, (15, 25 + th - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

def put_hud(frame, fps, model_name):
    text = f"{model_name} | {fps:.1f} FPS"
    cv2.putText(frame, text, (10, frame.shape[0] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

# --------- Prep I/O ---------
cap = cv2.VideoCapture(VIDEO_IN)
assert cap.isOpened(), f"Cannot open video: {VIDEO_IN}"
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (width, height))

# CSV header
csv_file = open(CSV_OUT, "w", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)
csv_writer.writerow([
    "frame", "time_sec", "track_id", "coco_class_id", "label",
    "confidence", "x1", "y1", "x2", "y2", "cx", "cy", "crossed", "direction"
])

# --------- Model ---------
model = YOLO(MODEL_WEIGHTS)

# Track state
last_centroid_side = {}             # track_id -> last side (+/-/0)
counted = set()                     # (track_id, direction_str)
count_A_to_B = 0
count_B_to_A = 0

# For FPS
t_last = time.time()
frame_idx = 0
ema_fps = None

# --------- Main loop via streaming tracker ---------
# persist=True keeps IDs across frames; classes filters to vehicles only
stream = model.track(
    source=VIDEO_IN,
    tracker=TRACKER,
    conf=CONF,
    iou=IOU,
    classes=VEHICLE_CLASS_IDS,
    persist=True,
    stream=True,       # yield Results per frame
    verbose=False
)

for results in stream:
    # results: ultralytics.engine.results.Results
    # original frame and plotted frame have same size
    frame = results.orig_img
    frame_idx += 1
    t_now = time.time()
    inst_fps = 1.0 / max(1e-6, (t_now - t_last))
    t_last = t_now
    ema_fps = inst_fps if ema_fps is None else 0.9 * ema_fps + 0.1 * inst_fps

    # Access detections
    boxes = results.boxes
    annotated = results.plot()  # draw default boxes, labels, track IDs if present

    # Draw counting line + HUD
    draw_line_with_count(annotated, LINE_P1, LINE_P2, count_A_to_B, count_B_to_A)
    put_hud(annotated, ema_fps or inst_fps, os.path.basename(MODEL_WEIGHTS))

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        clss = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        ids   = boxes.id.cpu().numpy() if boxes.id is not None else np.array([-1]*len(boxes))

        ax, ay = LINE_P1
        bx, by = LINE_P2

        for (x1, y1, x2, y2), c, conf, tid in zip(xyxy, clss, confs, ids):
            if tid == -1:
                # tracker has not assigned an ID yet; skip counting but still log
                direction = ""
                crossed = 0
            else:
                cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                side_now = np.sign(line_side(cx, cy, ax, ay, bx, by))

                prev = last_centroid_side.get(tid, side_now)
                crossed = 0
                direction = ""

                # Detect sign change across the line
                if prev != 0 and side_now != 0 and np.sign(prev) != np.sign(side_now):
                    # Determine direction A->B or B->A by side sign
                    # If we define A-side as side<0 and B-side as side>0 (arbitrary but consistent)
                    direction = "AtoB" if prev < 0 and side_now > 0 else "BtoA"
                    key = (int(tid), direction)
                    if key not in counted:
                        counted.add(key)
                        crossed = 1
                        if direction == "AtoB":
                            count_A_to_B += 1
                        else:
                            count_B_to_A += 1

                last_centroid_side[tid] = side_now

            # Logging (time_sec based on frame/fps to keep it stable)
            t_sec = frame_idx / fps
            label = CLASS_LABELS.get(int(c), str(int(c)))
            csv_writer.writerow([
                frame_idx, f"{t_sec:.3f}", int(tid), int(c), label,
                float(conf), int(x1), int(y1), int(x2), int(y2),
                int((x1 + x2) / 2.0), int((y1 + y2) / 2.0), crossed, direction
            ])

            # Draw extra ID bubble & direction hint
            if tid != -1:
                cx, cy = int((x1 + x2) / 2.0), int((y1 + y2) / 2.0)
                cv2.circle(annotated, (cx, cy), 3, (255, 255, 255), -1)
                if crossed:
                    cv2.putText(annotated, f"{direction}", (cx + 6, cy - 6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

    writer.write(annotated)

# Cleanup
writer.release()
csv_file.close()

print(f"\nSaved video: {VIDEO_OUT}")
print(f"Saved CSV:   {CSV_OUT}")



Saved video: runs/traffic\annotated_1.mp4
Saved CSV:   runs/traffic\events_1.csv


Expand and Streamline, modify the events to just include the aggregable ones.

In [ ]:
import os
import csv
import time
import numpy as np
import cv2
from ultralytics import YOLO

# ---------------- Config ----------------
VIDEO_IN = "videos/british_highway_traffic.mp4"
MODEL_WEIGHTS = "yolo11s.pt"      # try yolo11m.pt if GPU headroom
TRACKER_CONFIG = "botsort.yaml"   # BoT-SORT tracker config
CONF = 0.25
IOU = 0.5

# Vehicle class IDs in COCO: 2 car, 3 motorcycle, 5 bus, 7 truck
VEHICLE_CLASS_IDS = [2, 3, 5, 7]
CLASS_LABELS = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

# Virtual counting line (pixels)
LINE_P1 = (200, 300)
LINE_P2 = (900, 300)

OUT_DIR = "runs/traffic"
os.makedirs(OUT_DIR, exist_ok=True)
idx = len(os.listdir(OUT_DIR)) // 2
VIDEO_OUT = os.path.join(OUT_DIR, f"annotated_{idx}.mp4")
CSV_OUT = os.path.join(OUT_DIR, f"events_{idx}.csv")

# ---------------- Helpers ----------------
def line_side(px, py, ax, ay, bx, by):
    """Returns cross product sign to determine line side"""
    return (bx - ax) * (py - ay) - (by - ay) * (px - ax)

def draw_line(frame, p1, p2, count_A_to_B, count_B_to_A):
    cv2.line(frame, p1, p2, (0, 255, 255), 2)
    label = f"A→B: {count_A_to_B} | B→A: {count_B_to_A}"
    (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (10,10),(20+tw,20+th),(0,0,0),-1)
    cv2.putText(frame, label, (15, 25+th-12), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255),2)

def put_hud(frame, fps, model_name):
    text = f"{model_name} | {fps:.1f} FPS"
    cv2.putText(frame, text, (10, frame.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255),2)

# ---------------- Event Engine ----------------
class LineCrossCounter:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2
        self.last_side = {}    # vehicle_id -> last side
        self.counted = set()
        self.count_A_to_B = 0
        self.count_B_to_A = 0

    def check_cross(self, obj):
        x1, y1, x2, y2 = obj["bbox"]
        cx, cy = (x1 + x2)/2, (y1 + y2)/2
        side_now = np.sign(line_side(cx, cy, *self.p1, *self.p2))
        prev = self.last_side.get(obj["vehicle_id"], side_now)
        crossed, direction = 0, ""
        if prev != 0 and side_now != 0 and np.sign(prev) != np.sign(side_now):
            direction = "A->B" if prev < 0 and side_now > 0 else "B->A"
            key = (obj["vehicle_id"], direction)
            if key not in self.counted:
                self.counted.add(key)
                crossed = 1
                if direction == "A->B":
                    self.count_A_to_B += 1
                else:
                    self.count_B_to_A += 1
        self.last_side[obj["vehicle_id"]] = side_now
        return crossed, direction

# ---------------- CSV Logger ----------------
class CSVLogger:
    def __init__(self, path):
        self.file = open(path, "w", newline="", encoding="utf-8")
        self.writer = csv.writer(self.file)
        self.writer.writerow(["frame","time_sec","vehicle_id","class","label",
                              "confidence","x1","y1","x2","y2","cx","cy","crossed","direction"])
    def log(self, frame_idx, fps, obj, crossed=0, direction=""):
        x1,y1,x2,y2 = obj["bbox"]
        cid = obj["class"]
        label = CLASS_LABELS.get(cid, str(cid))
        conf = obj["conf"]
        cx, cy = (x1+x2)/2, (y1+y2)/2
        self.writer.writerow([frame_idx, f"{frame_idx/fps:.3f}", obj["vehicle_id"],
                              cid,label,float(conf),x1,y1,x2,y2,int(cx),int(cy),crossed,direction])
    def close(self):
        self.file.close()

# ---------------- Main Pipeline ----------------
# Initialize video capture to get frame properties
cap = cv2.VideoCapture(VIDEO_IN)
assert cap.isOpened(), f"Cannot open video: {VIDEO_IN}"
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

writer = cv2.VideoWriter(VIDEO_OUT, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

# Initialize modules
model = YOLO(MODEL_WEIGHTS)
counter = LineCrossCounter(LINE_P1, LINE_P2)
logger = CSVLogger(CSV_OUT)

# Track FPS
t_last = time.time()
ema_fps = None
frame_idx = 0

# ---------------- Streaming Tracker ----------------
stream = model.track(
    source=VIDEO_IN,
    tracker=TRACKER_CONFIG,
    conf=CONF,
    iou=IOU,
    classes=VEHICLE_CLASS_IDS,
    persist=True,
    stream=True,
    verbose=False
)

for results in stream:
    frame = results.orig_img
    frame_idx += 1

    # FPS estimate
    t_now = time.time()
    inst_fps = 1.0 / max(1e-6, t_now - t_last)
    t_last = t_now
    ema_fps = inst_fps if ema_fps is None else 0.9*ema_fps + 0.1*inst_fps

    # Access tracked detections
    boxes = results.boxes
    annotated = results.plot()  # draw boxes + IDs

    if boxes is not None and len(boxes) > 0:
        xyxy = boxes.xyxy.cpu().numpy()
        clss = boxes.cls.cpu().numpy().astype(int)
        confs = boxes.conf.cpu().numpy()
        ids   = boxes.id.cpu().numpy() if boxes.id is not None else np.array([-1]*len(boxes))

        for (x1, y1, x2, y2), c, conf, tid in zip(xyxy, clss, confs, ids):
            obj = {"vehicle_id": int(tid), "bbox": [x1,y1,x2,y2], "class": int(c), "conf": float(conf)}

            crossed, direction = (0, "") if tid==-1 else counter.check_cross(obj)
            logger.log(frame_idx, fps, obj, crossed, direction)

            # Draw centroid + direction hint
            if tid != -1:
                cx, cy = int((x1+x2)/2), int((y1+y2)/2)
                cv2.circle(annotated, (cx, cy), 3, (255, 255, 255), -1)
                if crossed:
                    cv2.putText(annotated, direction, (cx+6, cy-6),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Overlay line + counters + HUD
    draw_line(annotated, LINE_P1, LINE_P2, counter.count_A_to_B, counter.count_B_to_A)
    put_hud(annotated, ema_fps or inst_fps, os.path.basename(MODEL_WEIGHTS))
    writer.write(annotated)

# ---------------- Cleanup ----------------
writer.release()
logger.close()
print(f"\nSaved annotated video: {VIDEO_OUT}")
print(f"Saved CSV: {CSV_OUT}")
